# MMP9 Inhibitor — Data Collection & Cleaning

**Target:** Matrix Metalloproteinase-9 (MMP9) — ChEMBL ID: CHEMBL321  
**Objective:** Build a binary classification dataset to predict MMP9 inhibitor activity  
**Data Source:** ChEMBL database (bioactivity data from published literature)

---

## Background

MMP9 is a zinc-dependent endopeptidase involved in tumor invasion, angiogenesis, and metastasis.  
Inhibiting MMP9 is a therapeutic strategy for cancer treatment.  
This pipeline fetches experimentally measured IC50 values and creates clean binary labels for ML modeling.

**Pipeline Steps:**
## Workflow

+ Project Setup & Dependency Management
+ Target Identification (MMP9)
+ Raw Bioactivity Data Retrieval
+ Initial Data Integrity Filtering
+ Unit Standardization & pIC50 Transformation
+ Bioactivity Thresholding & Classification
+ Intermediate Compound Removal
+ Lipinski’s Rule of 5 Calculation
+ ADME Profiling & Pharmacokinetics
+ Chemical Structure Validation
+ Exploratory Data Analysis (EDA)
+ Data Export for Feature Engineering

**Note:** Class imbalance, Balanced Weights, scaffold splitting, and feature generation are handled in the next notebook

## 1. Imports

In [1]:
# ChEMBL client to fetch bioactivity data
from chembl_webresource_client.new_client import new_client

# Core data handling
import pandas as pd
import numpy as np

# Visualization libraries for EDA
import matplotlib.pyplot as plt
import seaborn as sns

# RDKit for cheminformatics (SMILES → molecular features)
from rdkit.Chem import MolFromSmiles, Descriptors, Draw, AllChem

# Dimensionality reduction
from sklearn.decomposition import PCA

# Prevent large warning prints
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

## 2. Target Search

Search ChEMBL for MMP9 to confirm the correct target ID.  
We specifically want **Homo sapiens** (human) MMP9 — ChEMBL ID: **CHEMBL321**  

Why human only? MMP9 from rat (CHEMBL3870) or mouse (CHEMBL2214) may have different binding characteristics  
and would reduce the biological relevance of our model for human drug discovery.

In [2]:
# Search for all MMP9 targets across species
target = new_client.target
target_query = target.search('mmp9')
targets = pd.DataFrame.from_dict(target_query)

# Display to confirm CHEMBL321 = Homo sapiens MMP9
targets[['organism', 'pref_name', 'target_chembl_id', 'target_type']].head()

,organism,pref_name,target_chembl_id,target_type
0,Rattus norvegicus,Matrix metalloproteinase-9,CHEMBL3870,SINGLE PROTEIN
1,Mus musculus,Matrix metalloproteinase-9,CHEMBL2214,SINGLE PROTEIN
2,Bos taurus,Matrix metalloproteinase-9,CHEMBL5846,SINGLE PROTEIN
3,Homo sapiens,Matrix metalloproteinase-9,CHEMBL321,SINGLE PROTEIN
4,Homo sapiens,Matrix metalloproteinase 2/9,CHEMBL3885505,PROTEIN FAMILY


## 3. Fetch Bioactivity Data

Fetch IC50 measurements for CHEMBL321 (human MMP9).  

**Why IC50?**  
IC50 (half-maximal inhibitory concentration) is the most common bioactivity measure in ChEMBL  
for enzyme inhibitors. It represents the compound concentration required to inhibit 50% of MMP9 activity.

**Why not Ki or Kd?**  
IC50 provides the largest dataset for MMP9. Ki/Kd will be explored separately in experiments.

In [3]:
# Fetch all IC50 bioactivity data for human MMP9
res = new_client.activity
res_query = res.filter(
    target_chembl_id='CHEMBL321'
).filter(
    standard_type='IC50'
)
df = pd.DataFrame.from_dict(res_query)
print(f"Raw records fetched: {len(df)}")
df.head()

Raw records fetched: 3964


,action_type,activity_comment,activity_id,activity_properties,assay_chembl_id,assay_description,assay_type,assay_variant_accession,assay_variant_mutation,bao_endpoint,...,target_organism,target_pref_name,target_tax_id,text_value,toid,type,units,uo_units,upper_value,value
0,None,None,33892,[],CHEMBL715225,In vitro inhibitory activity against matrix me...,B,None,None,BAO_0000190,...,Homo sapiens,Matrix metalloproteinase-9,9606,None,None,IC50,nM,UO_0000065,None,34.0
1,None,None,35115,[],CHEMBL715225,In vitro inhibitory activity against matrix me...,B,None,None,BAO_0000190,...,Homo sapiens,Matrix metalloproteinase-9,9606,None,None,IC50,nM,UO_0000065,None,9.0
2,None,None,35120,[],CHEMBL715225,In vitro inhibitory activity against matrix me...,B,None,None,BAO_0000190,...,Homo sapiens,Matrix metalloproteinase-9,9606,None,None,IC50,nM,UO_0000065,None,20.0
3,None,None,35125,[],CHEMBL715225,In vitro inhibitory activity against matrix me...,B,None,None,BAO_0000190,...,Homo sapiens,Matrix metalloproteinase-9,9606,None,None,IC50,nM,UO_0000065,None,3.0
4,None,None,35129,[],CHEMBL715225,In vitro inhibitory activity against matrix me...,B,None,None,BAO_0000190,...,Homo sapiens,Matrix metalloproteinase-9,9606,None,None,IC50,nM,UO_0000065,None,9.0


## 4. Data Quality Filters

Apply sequential filters to retain only high-quality, unambiguous data.

### Filter 1: Standard Relation  
Only keep exact measurements (relation = '=').  
Inequality measurements (>, <, <=) represent compounds where the true IC50 is unknown —  
labeling them as active or inactive would introduce noise.

### Filter 2: Assay Type  
Only keep binding assays (assay_type = 'B').  
Binding assays directly measure compound-protein interaction.  
Functional assays (A) measure downstream effects and are less direct.

### Filter 3: Null Removal  
Remove rows missing pChEMBL value or SMILES string —  
both are required for labeling and feature generation respectively.

In [4]:
# Inspect standard_type distribution (should be all IC50)
print("Standard type distribution:")
print(df['standard_type'].value_counts())

Standard type distribution:
standard_type
IC50    3964
Name: count, dtype: int64


In [5]:
# Inspect measurement relation types
print("Standard relation distribution:")
print(df['standard_relation'].value_counts())
print("\nRetaining only exact measurements (relation = '=')")

Standard relation distribution:
standard_relation
=     2464
>      461
<       16
<=       2
~        1
Name: count, dtype: int64

Retaining only exact measurements (relation = '=')


In [6]:
# Filter 1: Keep exact measurements only
before = len(df)
df = df[df['standard_relation'] == '=']
print(f"After relation filter: {before} → {len(df)} rows ({before - len(df)} removed)")

# Filter 2: Remove missing pChEMBL and SMILES
before = len(df)
df = df.dropna(subset=['pchembl_value', 'canonical_smiles'])
print(f"After null filter: {before} → {len(df)} rows ({before - len(df)} removed)")

df['pchembl_value'] = pd.to_numeric(df['pchembl_value'], errors='coerce')

# Create binary activity label
# pChEMBL >= 5.4 (IC50 <= 5uM) = ACTIVE (1)
# pChEMBL < 5.4 (IC50 > 5uM) = INACTIVE (0)
df['bioactivity_class'] = (df['pchembl_value'] >= 6).astype(int)
print(f"\nBioactivity class distribution:")
print(df['bioactivity_class'].value_counts())
print(f"\nNote: Class imbalance ({df['bioactivity_class'].sum()} active vs {(df['bioactivity_class']==0).sum()} inactive) will be addressed in notebook")

After relation filter: 3964 → 2464 rows (1500 removed)
After null filter: 2464 → 2411 rows (53 removed)

Bioactivity class distribution:
bioactivity_class
1    1760
0     651
Name: count, dtype: int64

Note: Class imbalance (1760 active vs 651 inactive) will be addressed in notebook


In [7]:
# Filter 3: Assay type distribution
print("Assay type distribution:")
print(df['assay_type'].value_counts())
print("\nRetaining only binding assays (B) — direct protein-compound interaction measurement")

Assay type distribution:
assay_type
B    2397
A      14
Name: count, dtype: int64

Retaining only binding assays (B) — direct protein-compound interaction measurement


In [8]:
# Apply assay type filter
before = len(df)
df = df[df['assay_type'] == 'B']
print(f"After assay type filter: {before} → {len(df)} rows ({before - len(df)} removed)")
print(f"\nFinal bioactivity distribution:")
print(df['bioactivity_class'].value_counts())

After assay type filter: 2411 → 2397 rows (14 removed)

Final bioactivity distribution:
bioactivity_class
1    1752
0     645
Name: count, dtype: int64


## 8. Save Clean Dataset

In [9]:
# Save to processed data folder
output_path = '../data/processed/mmp9_clean.csv'
df.to_csv(output_path, index=False)

print(f"Clean dataset saved to: {output_path}")
print(f"Final shape: {df.shape}")
print(f"Active compounds: {df['bioactivity_class'].sum()}")
print(f"Inactive compounds: {(df['bioactivity_class']==0).sum()}")
print(f"Active compounds percentage: {(df['bioactivity_class'].value_counts(normalize=True)[1]*100):.2f}")
print(f"Inactive compounds percentage: {(df['bioactivity_class'].value_counts(normalize=True)[0]*100):.2f}")
print("Data is highly imbalance and need to either reduce or balance weight to deal")

Clean dataset saved to: ../data/processed/mmp9_clean.csv
Final shape: (2397, 47)
Active compounds: 1752
Inactive compounds: 645
Active compounds percentage: 73.09
Inactive compounds percentage: 26.91
Data is highly imbalance and need to either reduce or balance weight to deal
